In [ ]:
import pandas as pd
import psycopg2
import numpy as np

# Configuración de base de datos (utiliza tus credenciales reales)
DB_CONFIG = {
    "dbname": "mydb",
    "user": "myuser",
    "password": "mypassword",
    "host": "localhost",
    "port": 5433
}

# Conectar a la base de datos PostgreSQL
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
print("Conexión a la base de datos establecida.")


Conexión a la base de datos establecida.


In [ ]:
# Leer el archivo CSV completo
csv_path = 'data/raw/data1.csv'
_df = pd.read_csv(csv_path)

# Limpiar espacios en los nombres de las columnas
_df.columns = _df.columns.str.strip()

# Limpiar espacios en los strings de MUESTRA
_df['MUESTRA'] = _df['MUESTRA'].astype(str).str.strip()

# Formatear la hora y fecha
_df['time_hhmm'] = pd.to_datetime(_df['time'].astype(str).str.replace('.', ':'), format='mixed').dt.strftime('%H:%M')
_df['date'] = pd.to_datetime(_df['date'], format='mixed').dt.strftime('%Y-%m-%d')

# Manejo de valores nulos o vacíos ("-" a None)
_df = _df.replace({np.nan: None, '-': None})

# --- CORRECCIÓN ---
# 1. Filtra el DataFrame y guárdalo en una nueva variable con un nombre descriptivo.
df_filtrado = _df[_df['date'].str.contains(r'2026-03-', regex=True)]
print("--- Filas que se procesarán (solo Diciembre 2022) ---")
display(df_filtrado) # Muestra el contenido del DataFrame que sí se usará

--- Filas que se procesarán (solo Diciembre 2022) ---


,date,time,tara,tweight,dweight,pweight,chemical_id,MUESTRA,pFe,pCu,pZn,pMo,pIns,pSol,time_hhmm
263,2023-06-02,10:20:00,109.0,1046.,207.5,937.0,5296.0,Relave Final,3.684,0.036,0.0,0.003,82.250,22.15,10:20
264,2023-06-02,10:21:00,110.0,914.,137.5,804.0,5297.0,Concentrado Colectivo,29.5,22.39,0.373,1.281,10.400,17.10,10:21
265,2023-06-02,10:24:00,109.5,1209.5,396.5,1100.0,5298.0,Concentrado Primario L2,25.62,13.33,0.195,0.587,27.780,36.05,10:24
266,2023-06-02,10:27:00,108.0,1230.,362.0,1122.0,5294.0,Rebose Hidrociclones L1,4.23,0.345,0.007,0.013,82.280,32.26,10:27
267,2023-06-02,10:51:00,110.0,1245.5,375.5,1135.5,5295.0,Rebose Hidrociclones L2,3.991,0.23,0.006,0.011,81.790,33.07,10:51
268,2023-06-05,10:32:00,110.5,1140.5,244.0,1030.0,5300.0,Relave Final,2.664,0.061,0.002,0.003,82.590,23.69,10:32
269,2023-06-05,10:36:00,110.5,1102.,292.5,991.5,5299.0,Rebose Hidrociclones L1,3.126,0.349,0.006,0.014,80.690,29.50,10:36
270,2023-06-05,10:37:00,111.0,1213.,346.5,1102.0,5302.0,Concentrado Primario L2,24.95,12.89,0.108,0.578,28.720,31.44,10:37
271,2023-06-05,10:41:00,110.5,1165.,306.0,1054.5,5303.0,Concentrado Primario L1,20.06,7.33,0.076,0.449,43.520,29.02,10:41
272,2023-06-05,10:42:00,111.0,934.5,212.5,823.5,5301.0,Concentrado Colectivo,29.91,22.4,0.0,1.104,9.280,25.80,10:42


In [45]:
ids_permitidos = (17, 18, 19, 20, 21, 22, 23, 24, 25, 26)

# 2. Haz que el bucle 'for' itere sobre el DataFrame filtrado.
for index, row in df_filtrado.iterrows():
    csv_date = row['date']
    csv_time = row['time_hhmm']
    csv_muestra = str(row['MUESTRA']).strip()

    query_buscar_fecha_hora = "SELECT id, sample_id FROM works4cdp_assay WHERE date = %s AND to_char(time, 'HH24:MI') = %s AND sample_id IN %s;"
    cursor.execute(query_buscar_fecha_hora, (csv_date, csv_time, ids_permitidos))

    ensayos_encontrados = cursor.fetchall()
    print(f"--- Evaluando Fila {index} ---")
    print(f"Ensayos encontrados a las {csv_time} del {csv_date}: {ensayos_encontrados}")

    fue_actualizado = False

    for assay_id, sample_id in ensayos_encontrados:
        
        query_buscar_muestra = "SELECT name, tag FROM works4cdp_sample WHERE id = %s;"
        cursor.execute(query_buscar_muestra, (sample_id,))
        datos_muestra_bd = cursor.fetchone()
        
        if datos_muestra_bd:
            db_muestra_nombre = str(datos_muestra_bd[0]).strip()
            
            if db_muestra_nombre == csv_muestra:
                print(f"¡Coincidencia de Muestra encontrada! Evaluando UPDATE para ID Assay: {assay_id}...")
                
                update_query = """
                    UPDATE works4cdp_assay
                    SET 
                        tara = %s, tweight = %s, dweight = %s, pweight = %s,
                        chemical_id = %s, "pFe" = %s, "pCu" = %s, "pZn" = %s,
                        "pMo" = %s, "pIns" = %s, "pSol" = %s
                    WHERE id = %s
                    AND (
                        tara IS DISTINCT FROM %s OR
                        tweight IS DISTINCT FROM %s OR
                        dweight IS DISTINCT FROM %s OR
                        pweight IS DISTINCT FROM %s OR
                        chemical_id IS DISTINCT FROM %s OR
                        "pFe" IS DISTINCT FROM %s OR
                        "pCu" IS DISTINCT FROM %s OR
                        "pZn" IS DISTINCT FROM %s OR
                        "pMo" IS DISTINCT FROM %s OR
                        "pIns" IS DISTINCT FROM %s OR
                        "pSol" IS DISTINCT FROM %s
                    );
                """
                
                # Preparamos los valores a actualizar y comparar
                valores_csv = (
                    float(row['tara']) if row['tara'] is not None else None,
                    float(row['tweight']) if row['tweight'] is not None else None,
                    float(row['dweight']) if row['dweight'] is not None else None,
                    float(row['pweight']) if row['pweight'] is not None else None,
                    int(row['chemical_id']) if row['chemical_id'] is not None else None,
                    float(row['pFe']) if row['pFe'] is not None else None,
                    float(row['pCu']) if row['pCu'] is not None else None,
                    float(row['pZn']) if row['pZn'] is not None else None,
                    float(row['pMo']) if row['pMo'] is not None else None,
                    float(row['pIns']) if row['pIns'] is not None else None,
                    float(row['pSol']) if row['pSol'] is not None else None
                )
                
                # Parámetros SET (11) + WHERE id (1) + AND IS DISTINCT FROM (11)
                valores_actualizacion = valores_csv + (int(assay_id),) + valores_csv
                
                try:
                    cursor.execute(update_query, valores_actualizacion)  
                    conn.commit()  
                    
                    if cursor.rowcount > 0:
                        print("✅ Fila actualizada exitosamente en la base de datos.\n")
                    else:
                        print("⏭️ Fila omitida (datos idénticos, no requirió actualización).\n")

                    fue_actualizado = True
                    break
                    
                except Exception as e:
                    print(f"❌ Error al actualizar la base de datos: {e}\n")
                    conn.rollback()

    if not fue_actualizado:
        print("⚠️ No se logró actualizar el ensayo.\n")

# Cerramos la conexión para liberar recursos
cursor.close()
conn.close()
print("Conexión cerrada.")

--- Evaluando Fila 263 ---
Ensayos encontrados a las 10:20 del 2023-06-02: [(13001, 19)]
¡Coincidencia de Muestra encontrada! Evaluando UPDATE para ID Assay: 13001...
⏭️ Fila omitida (datos idénticos, no requirió actualización).

--- Evaluando Fila 264 ---
Ensayos encontrados a las 10:21 del 2023-06-02: [(12224, 24)]
¡Coincidencia de Muestra encontrada! Evaluando UPDATE para ID Assay: 12224...
⏭️ Fila omitida (datos idénticos, no requirió actualización).

--- Evaluando Fila 265 ---
Ensayos encontrados a las 10:24 del 2023-06-02: [(11626, 25)]
¡Coincidencia de Muestra encontrada! Evaluando UPDATE para ID Assay: 11626...
⏭️ Fila omitida (datos idénticos, no requirió actualización).

--- Evaluando Fila 266 ---
Ensayos encontrados a las 10:27 del 2023-06-02: [(10175, 17)]
¡Coincidencia de Muestra encontrada! Evaluando UPDATE para ID Assay: 10175...
⏭️ Fila omitida (datos idénticos, no requirió actualización).

--- Evaluando Fila 267 ---
Ensayos encontrados a las 10:51 del 2023-06-02: [(139